# Common Logging Utilities

**Purpose:** Standardized logging configuration and utilities for all pipeline notebooks.

**Usage:**
```python
%run "./common/common_logging"

logger = get_logger(__name__)
logger.info("Starting process")
```

**Features:**
* Consistent log formatting across all notebooks
* Configurable log levels
* Performance timing decorators
* Exception logging helpers
* Structured logging support

In [0]:
import logging
import functools
import time
from typing import Any, Callable, Optional
from datetime import datetime

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================

# Default log format
DEFAULT_LOG_FORMAT = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
DEFAULT_DATE_FORMAT = '%Y-%m-%d %H:%M:%S'
DEFAULT_LOG_LEVEL = logging.INFO

# Track whether logging has been configured globally
_logging_configured = False

In [0]:
def configure_logging(
    level: int = DEFAULT_LOG_LEVEL,
    log_format: str = DEFAULT_LOG_FORMAT,
    date_format: str = DEFAULT_DATE_FORMAT
) -> None:
    """
    Configure global logging settings.
    
    Args:
        level: Logging level (e.g., logging.INFO, logging.DEBUG)
        log_format: Format string for log messages
        date_format: Format string for timestamps
    
    Example:
        configure_logging(level=logging.DEBUG)
    """
    global _logging_configured
    
    if not _logging_configured:
        logging.basicConfig(
            level=level,
            format=log_format,
            datefmt=date_format
        )
        _logging_configured = True

In [0]:
def get_logger(
    name: str,
    level: Optional[int] = None
) -> logging.Logger:
    """
    Get a configured logger instance.
    
    Args:
        name: Logger name (typically __name__)
        level: Optional log level override
    
    Returns:
        Configured logger instance
    
    Example:
        logger = get_logger(__name__)
        logger.info("Processing started")
    """
    # Ensure global logging is configured
    if not _logging_configured:
        configure_logging()
    
    logger = logging.getLogger(name)
    
    if level is not None:
        logger.setLevel(level)
    
    return logger

In [0]:
def log_execution_time(func: Callable) -> Callable:
    """
    Decorator to log function execution time.
    
    Example:
        @log_execution_time
        def process_data():
            # ... processing logic
            pass
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger = get_logger(func.__module__)
        start_time = time.time()
        
        logger.info(f"Starting {func.__name__}")
        
        try:
            result = func(*args, **kwargs)
            execution_time = time.time() - start_time
            logger.info(f"Completed {func.__name__} in {execution_time:.2f}s")
            return result
        except Exception as e:
            execution_time = time.time() - start_time
            logger.error(
                f"Failed {func.__name__} after {execution_time:.2f}s: {str(e)}"
            )
            raise
    
    return wrapper

In [0]:
def log_exception(
    logger: logging.Logger,
    exception: Exception,
    context: str = ""
) -> None:
    """
    Log exception with full context and traceback.
    
    Args:
        logger: Logger instance
        exception: Exception to log
        context: Additional context string
    
    Example:
        try:
            process_data()
        except Exception as e:
            log_exception(logger, e, "During data processing")
            raise
    """
    import traceback
    
    error_msg = f"{context}: {str(exception)}" if context else str(exception)
    logger.error(error_msg)
    logger.debug(traceback.format_exc())

In [0]:
def log_section_start(logger: logging.Logger, section_name: str, width: int = 80) -> None:
    """
    Log a formatted section header.
    
    Args:
        logger: Logger instance
        section_name: Name of the section
        width: Width of the separator line
    
    Example:
        log_section_start(logger, "Data Validation")
    """
    separator = "=" * width
    logger.info(separator)
    logger.info(section_name.upper().center(width))
    logger.info(separator)

def log_section_end(logger: logging.Logger, section_name: str, width: int = 80) -> None:
    """
    Log a formatted section footer.
    
    Args:
        logger: Logger instance
        section_name: Name of the section
        width: Width of the separator line
    
    Example:
        log_section_end(logger, "Data Validation")
    """
    separator = "=" * width
    logger.info(f"Completed: {section_name}")
    logger.info(separator)

In [0]:
def log_metrics(
    logger: logging.Logger,
    metrics: dict,
    prefix: str = "Metrics"
) -> None:
    """
    Log structured metrics in a readable format.
    
    Args:
        logger: Logger instance
        metrics: Dictionary of metric name -> value
        prefix: Prefix for the log message
    
    Example:
        log_metrics(logger, {
            "rows_processed": 10000,
            "duration_seconds": 45.2,
            "success_rate": 0.98
        })
    """
    logger.info(f"{prefix}:")
    for key, value in metrics.items():
        if isinstance(value, float):
            logger.info(f"  {key}: {value:.2f}")
        elif isinstance(value, int) and value > 1000:
            logger.info(f"  {key}: {value:,}")
        else:
            logger.info(f"  {key}: {value}")

def log_dataframe_info(
    logger: logging.Logger,
    df,
    name: str = "DataFrame"
) -> None:
    """
    Log PySpark DataFrame information.
    
    Args:
        logger: Logger instance
        df: PySpark DataFrame
        name: Name/description of the DataFrame
    
    Example:
        log_dataframe_info(logger, customers_df, "Customers")
    """
    row_count = df.count()
    col_count = len(df.columns)
    logger.info(f"{name}: {row_count:,} rows × {col_count} columns")

In [0]:
def log_audit_pipeline_run(
    pipeline_name: str,
    run_id: str,
    status: str,
    start_time: datetime,
    end_time: datetime = None,
    rows_read: int = 0,
    rows_written: int = 0,
    error_message: str = None,
    catalog: str = "workspace",
    schema: str = "audit"
) -> None:
    """
    Log pipeline run audit entry to audit.audit_pipeline_runs.
    
    Args:
        pipeline_name: Name of the pipeline (e.g., 'silver_processing')
        run_id: Unique run identifier
        status: Run status ('SUCCESS', 'FAILED', 'RUNNING')
        start_time: Pipeline start timestamp
        end_time: Pipeline end timestamp (None if still running)
        rows_read: Number of rows read
        rows_written: Number of rows written
        error_message: Error message if failed
        catalog: Catalog name (default: workspace)
        schema: Schema name (default: audit)
    
    Example:
        log_audit_pipeline_run(
            pipeline_name='silver_processing',
            run_id='12345',
            status='SUCCESS',
            start_time=datetime.now(),
            rows_written=10000
        )
    """
    from pyspark.sql import Row
    from decimal import Decimal
    import time
    
    # Calculate runtime
    if end_time is None:
        end_time = datetime.now()
    runtime_seconds = (end_time - start_time).total_seconds()
    
    # Create audit record
    audit_record = Row(
        audit_run_sk=int(time.time() * 1000),
        run_id=run_id,
        pipeline_name=pipeline_name,
        environment='PROD',
        start_time=start_time,
        end_time=end_time,
        status=status,
        rows_read=rows_read,
        rows_written=rows_written,
        runtime_seconds=Decimal(str(round(runtime_seconds, 2))),
        error_message=error_message
    )
    
    # Write to audit table with explicit schema
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, DecimalType
    
    audit_schema = StructType([
        StructField("audit_run_sk", LongType(), False),
        StructField("run_id", StringType(), False),
        StructField("pipeline_name", StringType(), False),
        StructField("environment", StringType(), False),
        StructField("start_time", TimestampType(), False),
        StructField("end_time", TimestampType(), True),
        StructField("status", StringType(), False),
        StructField("rows_read", LongType(), True),
        StructField("rows_written", LongType(), True),
        StructField("runtime_seconds", DecimalType(18, 2), True),
        StructField("error_message", StringType(), True)
    ])
    
    audit_df = spark.createDataFrame([audit_record], schema=audit_schema)
    target_table = f"{catalog}.{schema}.audit_pipeline_runs"
    audit_df.write.mode('append').saveAsTable(target_table)
    
    # Log using logger
    logger = get_logger(__name__)
    if status == 'SUCCESS':
        log_metrics(logger, {
            "Pipeline": pipeline_name,
            "Rows Read": rows_read,
            "Rows Written": rows_written,
            "Runtime (s)": round(runtime_seconds, 2)
        }, prefix="Audit")
    elif status == 'FAILED':
        logger.error(f"Pipeline {pipeline_name} failed: {error_message}")